In [0]:
%sql
--Dimensão data_center, que é a mesma coisa que o data center silver

CREATE OR REPLACE TABLE portfolio_energia.gold.dim_data_center AS
SELECT 
    ds.id,
    ds.name,
    ds.operator,
    ds.region,
    ds.iso2
FROM    
    portfolio_energia.silver.datacenter_silver AS ds
WHERE id IS NOT NULL;

-- Criando Dimensão Indicador, que liga com World Bank

CREATE OR REPLACE TABLE portfolio_energia.gold.dim_indicator AS
SELECT DISTINCT
    md5(indicator_code) AS indicator_id,
    indicator_code,
    indicator_name
FROM (
    SELECT indicator_code, indicator_name FROM portfolio_energia.silver.world_bank_silver_populacao
    UNION
    SELECT indicator_code, indicator_name FROM portfolio_energia.silver.world_bank_silver_pfdi
    UNION
    SELECT indicator_code, indicator_name FROM portfolio_energia.silver.world_bank_silver_peletricidade
    UNION
    SELECT indicator_code, indicator_name FROM portfolio_energia.silver.world_bank_silver_pib_per_capita
)
WHERE indicator_code IS NOT NULL;
    
-- Criado a dimensão date

CREATE OR REPLACE TABLE portfolio_energia.gold.dim_date AS
SELECT DISTINCT 
    CAST(date_format(es.date, 'yyyyMMdd') AS INT) AS date_id,
    es.date as month_start_date,
    EXTRACT(MONTH FROM es.date) AS month,
    EXTRACT(YEAR FROM es.date) AS year,
    EXTRACT(QUARTER FROM es.date) AS quarter,
    date_format(es.date, 'MMMM') AS month_name_full,
    date_format(es.date, 'MMM') AS month_name_short
FROM    
    portfolio_energia.silver.ember_intensity_silver AS es
WHERE
    es.date IS NOT NULL;

-- Por fim, criando a dimensão location

CREATE OR REPLACE TABLE portfolio_energia.gold.dim_location AS
SELECT DISTINCT
    md5(lo.name) AS id_location,
    lo.name AS country,
    lo.`alpha-2` AS iso2,
    lo.`alpha-3` AS iso3
FROM    
    read_files(
        '/Volumes/portfolio_energia/bronze/csv_location/all.csv',
        format => 'csv',
        header => true
    ) AS lo  
WHERE lo.name IS NOT NULL;

